# 02 - Analisis Hasil Eksperimen

Notebook ini membaca berkas hasil pada `reports/results/` dan menyusun ulang
tabel serta grafik yang dipakai pada laporan. **Tidak ada training di sini** dan
tidak ada angka yang diketik manual: semua nilai berasal dari JSON hasil eksperimen.

Jalankan pipeline lebih dulu:

```bash
python scripts/run_experiments.py
```

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "reports" / "results"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load(name):
    path = RESULTS / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

baseline = load("baseline.json")
cross_user = load("cross_user.json")
improvement = load("improvement.json")
cross_device = load("cross_device.json")
print("baseline:", bool(baseline), "| cross_user:", bool(cross_user), "| improvement:", bool(improvement))

## Eksperimen A - baseline pada split resmi

In [ ]:
rows = []
for name, payload in (baseline or {}).get("models", {}).items():
    metrics = payload["metrics"]
    rows.append({
        "model": payload.get("display_name", name),
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "fit_time_s": payload.get("fit_time_s"),
        "predict_time_s": payload.get("predict_time_s"),
    })
baseline_table = pd.DataFrame(rows)
display(baseline_table.round(4))

In [ ]:
if not baseline_table.empty:
    ax = baseline_table.set_index("model")[["accuracy", "balanced_accuracy", "macro_f1"]].plot(
        kind="bar", figsize=(8, 4), color=["#8ecae6", "#219ebc", "#023047"]
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Skor")
    ax.set_title("Baseline pada split resmi berbasis subjek")
    plt.tight_layout()
    plt.savefig(FIGURES / "notebook_02_baseline.png", dpi=150)
    plt.show()

## Eksperimen B - generalisasi lintas pengguna

RF/SVM: Leave-One-Subject-Out. CNN: GroupKFold(6) karena biaya pelatihan CPU.

In [ ]:
rows = []
for name, payload in (cross_user or {}).get("models", {}).items():
    aggregate = payload.get("aggregate", {})
    per_subject = payload.get("per_subject_aggregate", {}).get("macro_f1", {})
    rows.append({
        "model": payload.get("display_name", name),
        "protokol": payload.get("protocol"),
        "folds": payload.get("n_folds"),
        "macro_f1_mean": aggregate.get("macro_f1", {}).get("mean"),
        "macro_f1_std": aggregate.get("macro_f1", {}).get("std"),
        "subject_min": per_subject.get("min"),
        "subject_max": per_subject.get("max"),
    })
cross_table = pd.DataFrame(rows)
display(cross_table.round(4))

In [ ]:
if cross_user:
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, payload in cross_user.get("models", {}).items():
        records = sorted(payload.get("per_subject", []), key=lambda item: item["subject"])
        if not records:
            continue
        ax.plot([r["subject"] for r in records], [r["macro_f1"] for r in records], marker="o", label=payload.get("display_name", name))
    ax.set_xlabel("Subjek yang diuji")
    ax.set_ylabel("Macro-F1")
    ax.set_ylim(0, 1.02)
    ax.set_title("Performa lintas pengguna per subjek")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "notebook_02_cross_user_per_subject.png", dpi=150)
    plt.show()

## Eksperimen D - strategi peningkatan generalisasi

In [ ]:
rows = []
for model_name, model_payload in (improvement or {}).get("strategies", {}).items():
    for strategy, record in model_payload.items():
        aggregate = record.get("aggregate", {})
        rows.append({
            "model": model_name,
            "strategy": strategy,
            "macro_f1_mean": aggregate.get("macro_f1", {}).get("mean"),
            "macro_f1_std": aggregate.get("macro_f1", {}).get("std"),
            "delta_vs_base": (improvement.get("deltas_vs_base", {}).get(model_name, {}).get(strategy, {}) or {}).get("macro_f1"),
        })
improvement_table = pd.DataFrame(rows)
display(improvement_table.round(4))

## Eksperimen C - lintas perangkat

Status di bawah dibaca apa adanya dari `cross_device.json`.

In [ ]:
print(json.dumps(cross_device, indent=2, ensure_ascii=False)[:1500] if cross_device else "cross_device.json belum ada")